# Change Event Strings

This notebook creates a turtle file with multilingual strings for all municipality change events.

## Download and Save AGVCH XML File

At the moment, the XML with version 1.1 of the eCH Standard 0071 is used (instead of the newer 1.2) because the existing data on LINDAS also use 1.1

In [1]:
import requests
import zipfile
import io
import os

response = requests.post("https://www.agvchapp.bfs.admin.ch/file/xml")
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in the ZIP archive:", z.namelist())
    for file_info in z.filelist:
        # Check if path matches pattern: */1.1 (last update)/*.xml
        path_parts = file_info.filename.split('/')
        if (len(path_parts) == 3 and 
            path_parts[1] == '1.1  (last update)' and 
            path_parts[2].endswith('.xml')):
            
            # Extract just the XML file with a cleaner name
            xml_content = z.read(file_info.filename)
            output_path = os.path.join("../data/xml", path_parts[2])
            os.makedirs("../data/xml", exist_ok=True)
            
            with open(output_path, 'wb') as f:
                f.write(xml_content)
            
            print(f"Extracted: {file_info.filename} -> {output_path}")
            break  # Stop after finding the first match

Files in the ZIP archive: ['dz-b-00.04-hgv-02.20260101/', 'dz-b-00.04-hgv-02.20260101/1.1  (last update)/', 'dz-b-00.04-hgv-02.20260101/1.1  (last update)/eCH-0071-1-1.xsd', 'dz-b-00.04-hgv-02.20260101/1.1  (last update)/eCH0071_260101.xml', 'dz-b-00.04-hgv-02.20260101/1.2.0/', 'dz-b-00.04-hgv-02.20260101/1.2.0/eCH-0071-2-0.xsd', 'dz-b-00.04-hgv-02.20260101/1.2.0/eCH0071_260101.xml', 'dz-b-00.04-hgv-02.20260101/readme.txt']
Extracted: dz-b-00.04-hgv-02.20260101/1.1  (last update)/eCH0071_260101.xml -> ../data/xml/eCH0071_260101.xml


## Create Python Dataframe for Municipalities

In [2]:
import xmltodict
import pandas as pd

data = xmltodict.parse(xml_content)

munies = pd.DataFrame(data["eCH-0071:nomenclature"]["municipalities"]["municipality"])
display(munies.head())

,historyMunicipalityId,districtHistId,cantonAbbreviation,municipalityId,municipalityLongName,municipalityShortName,municipalityEntryMode,municipalityStatus,municipalityAdmissionNumber,municipalityAdmissionMode,municipalityAdmissionDate,municipalityAbolitionNumber,municipalityAbolitionMode,municipalityAbolitionDate,municipalityDateOfChange
0,10001,10189,GR,3501,Alvaschein,Alvaschein,11,1,1000,20,1960-01-01,1944,24,2000-12-31,2000-12-31
1,10002,10192,SG,3403,Ganterschwil,Ganterschwil,11,1,1000,20,1960-01-01,2253,24,2002-12-31,2002-12-31
2,10003,10189,GR,3523,Wiesen (GR),Wiesen (GR),11,1,1000,20,1960-01-01,1957,24,2000-12-31,2000-12-31
3,10004,10189,GR,3522,Filisur,Filisur,11,1,1000,20,1960-01-01,1956,24,2000-12-31,2000-12-31
4,10005,10189,GR,3521,Bergün/Bravuogn,Bergün/Bravuogn,11,1,1000,20,1960-01-01,1955,24,2000-12-31,2000-12-31


In [3]:
# first step replace municipalityAdmissionNumber = 1000 with municipalityAdmissionNumber = 1000_[historyMunicipalityId] (only for those rows where municipalityAdmissionNumber = 1000) - same as LINDAS does

munies["municipalityAdmissionNumber"] = munies.apply(
    lambda row: f"{row['municipalityAdmissionNumber']}_{row['historyMunicipalityId']}" if row['municipalityAdmissionNumber'] == "1000" else row['municipalityAdmissionNumber'], 
    axis=1)

display(munies.head())

,historyMunicipalityId,districtHistId,cantonAbbreviation,municipalityId,municipalityLongName,municipalityShortName,municipalityEntryMode,municipalityStatus,municipalityAdmissionNumber,municipalityAdmissionMode,municipalityAdmissionDate,municipalityAbolitionNumber,municipalityAbolitionMode,municipalityAbolitionDate,municipalityDateOfChange
0,10001,10189,GR,3501,Alvaschein,Alvaschein,11,1,1000_10001,20,1960-01-01,1944,24,2000-12-31,2000-12-31
1,10002,10192,SG,3403,Ganterschwil,Ganterschwil,11,1,1000_10002,20,1960-01-01,2253,24,2002-12-31,2002-12-31
2,10003,10189,GR,3523,Wiesen (GR),Wiesen (GR),11,1,1000_10003,20,1960-01-01,1957,24,2000-12-31,2000-12-31
3,10004,10189,GR,3522,Filisur,Filisur,11,1,1000_10004,20,1960-01-01,1956,24,2000-12-31,2000-12-31
4,10005,10189,GR,3521,Bergün/Bravuogn,Bergün/Bravuogn,11,1,1000_10005,20,1960-01-01,1955,24,2000-12-31,2000-12-31


## Creation of Event Dictionary

In [4]:
# dictionary structure:
# {
#   eventId: {
#       "predecessors": [
#           {
#               "id": historyMunicipalityId,
#               "name": municipalityLongName,
#               "mode": municipalityAbolitionMode
#           },
#           ...
#       ],
#       "successors": [
#           {
#               "id": historyMunicipalityId,
#               "name": municipalityLongName,
#               "mode": municipalityAdmissionMode
#           },
#           ...
#       ]
#   },
#   ...
# }
# attention: eventId may appear multiple times in the dataframe, sometimes as municipalityAdmissionNumber, sometimes as municipalityAbolitionNumber, predecessors may be None, successors not

def process_row(row, event_dict):

    # Admission
    
    # if admission not in event_dict, initialize it
    if row["municipalityAdmissionNumber"] not in event_dict:
        event_dict[row["municipalityAdmissionNumber"]] = {
            "successors": [],
            "predecessors": [],
            "abModes": set(),
            "adModes": set()
        }
    
    # successor always present
    successor = {
        "id": row["historyMunicipalityId"],
        "name": row["municipalityLongName"],
        "mode": row["municipalityAdmissionMode"]
    }

    # add successor in event's successors
    event_dict[row["municipalityAdmissionNumber"]]["successors"].append(successor)
    event_dict[row["municipalityAdmissionNumber"]]["adModes"].add(row["municipalityAdmissionMode"])

    # Abolition (may be None if it is a current version)

    # if abolition present
    if row["municipalityAbolitionNumber"] is not None:


        # if abolition not in event_dict, initialize it
        if row["municipalityAbolitionNumber"] not in event_dict:
            event_dict[row["municipalityAbolitionNumber"]] = {
                "successors": [],
                "predecessors": [],
                "abModes": set(),
                "adModes": set()
            }

    
        predecessor = {
            "id": row["historyMunicipalityId"],
            "name": row["municipalityLongName"],
            "mode": row["municipalityAbolitionMode"]
        }

        event_dict[row["municipalityAbolitionNumber"]]["predecessors"].append(predecessor)
        event_dict[row["municipalityAbolitionNumber"]]["abModes"].add(row["municipalityAbolitionMode"])

event_dict = {}

munies.apply(process_row, axis=1, args=(event_dict,))

0       None
1       None
2       None
3       None
4       None
        ... 
5863    None
5864    None
5865    None
5866    None
5867    None
Length: 5868, dtype: object

In [5]:
# example

event_dict["3927"]

{'successors': [{'id': '16135', 'name': 'Brugg', 'mode': '26'}],
 'predecessors': [{'id': '14376', 'name': 'Schinznach-Bad', 'mode': '29'},
  {'id': '15387', 'name': 'Brugg', 'mode': '26'}],
 'abModes': {'26', '29'},
 'adModes': {'26'}}

## String Creation for Change Event

In [6]:
# Multilingual helper function to join names with commas and in the last place "und"

def join_names(names, lang):
    
    #
    and_words = {
        "de": "und",
        "fr": "et",
        "it": "e",
        "en": "and"
    }
    
    match names:
        case []:
            return ""
        case [a]:
            return a
        case [a, b]:
            return f"{a} {and_words[lang]} {b}"
        case _:
            return f"{', '.join(names[:-1])} {and_words[lang]} {names[-1]}"

In [7]:
def create_strings(event):
    
    # Ersterfassung
    if event["adModes"] == {'20'}:
        event["changeType"] = "InitialRecording"
        event.setdefault("changeString", {})["de"] = f"Ersterfassung der Gemeinde {event['successors'][0]['name']}"
        event.setdefault("changeString", {})["fr"] = f"Enregistrement initial de la commune {event['successors'][0]['name']}"
        event.setdefault("changeString", {})["it"] = f"Prima registrazione del comune {event['successors'][0]['name']}"
        event.setdefault("changeString", {})["en"] = f"Initial recording of the municipality {event['successors'][0]['name']}"

    # Bezirks-/Kantonswechsel
    # Berücksichtigt nicht die Namen der Bezirke/Kantone, sondern nur die Namen der Gemeinden.
    if event["abModes"] == {'24'} and event["adModes"] == {'24'}:
        event["changeType"] = "DistrictCantonChange"
        new_name = event["successors"][0]["name"]
        event.setdefault("changeString", {})["de"] = f"Bezirks-/Kantonswechsel von {new_name}"
        event.setdefault("changeString", {})["fr"] = f"Changement de district/canton de {new_name}"
        event.setdefault("changeString", {})["it"] = f"Cambiamento di distretto/cantone da {new_name}"
        event.setdefault("changeString", {})["en"] = f"District/canton change to {new_name}"

    # Neunummerierung
    if event["abModes"] == {'27'} and event["adModes"] == {'27'}:
        event["changeType"] = "Renumbering"
        event.setdefault("changeString", {})["de"] = f"Neunummerierung Gemeinde/Bezirk für {event['successors'][0]['name']}."
        event.setdefault("changeString", {})["fr"] = f"Renumérotation de la commune/district pour {event['successors'][0]['name']}."
        event.setdefault("changeString", {})["it"] = f"Rinumerazione del comune/distritto per {event['successors'][0]['name']}."
        event.setdefault("changeString", {})["en"] = f"Renumbering of municipality/district for {event['successors'][0]['name']}."

    # Eingemeindung
    if event["abModes"] == {'26', '29'} and event["adModes"] == {'26'}:
        event["changeType"] = "Incorporation"
        new = []
        old = []
        for predecessor in event["predecessors"]:
            if predecessor["mode"] == '26':
                new.append(predecessor["name"])
            else:
                old.append(predecessor["name"])
        new = sorted(new)
        old = sorted(old)
        event.setdefault("changeString", {})["de"] = f"Eingemeindung von {join_names(old, 'de')} in {join_names(new, 'de')}"
        event.setdefault("changeString", {})["fr"] = f"Incoporation de {join_names(old, 'fr')} dans {join_names(new, 'fr')}"
        event.setdefault("changeString", {})["it"] = f"Incorporazione di {join_names(old, 'it')} in {join_names(new, 'it')}"
        event.setdefault("changeString", {})["en"] = f"Incorporation of {join_names(old, 'en')} into {join_names(new, 'en')}"

    # Fusion / Trennung
    if event["abModes"] == {'29'} and event["adModes"] == {'21'}:
        new = []
        old = []
        for predecessor in event["predecessors"]:
            old.append(predecessor["name"])
        for successor in event["successors"]:
            new.append(successor["name"])
        new = sorted(new)
        old = sorted(old)
        if len(old) == 1:
            event["changeType"] = "Separation"
            event.setdefault("changeString", {})["de"] = f"Trennung von {join_names(old, 'de')} in {join_names(new, 'de')}"
            event.setdefault("changeString", {})["fr"] = f"Séparation de {join_names(old, 'fr')} en {join_names(new, 'fr')}"
            event.setdefault("changeString", {})["it"] = f"Separazione di {join_names(old, 'it')} in {join_names(new, 'it')}"
            event.setdefault("changeString", {})["en"] = f"Separation of {join_names(old, 'en')} into {join_names(new, 'en')}"
        else:
            event["changeType"] = "Fusion"
            event.setdefault("changeString", {})["de"] = f"Fusion von {join_names(old, 'de')} zu {join_names(new, 'de')}"
            event.setdefault("changeString", {})["fr"] = f"Fusion de {join_names(old, 'fr')} en {join_names(new, 'fr')}"
            event.setdefault("changeString", {})["it"] = f"Fusione di {join_names(old, 'it')} in {join_names(new, 'it')}"
            event.setdefault("changeString", {})["en"] = f"Fusion of {join_names(old, 'en')} into {join_names(new, 'en')}"
    
    # Namensänderung
    if event["abModes"] == {'23'} and event["adModes"] == {'23'}:
        event["changeType"] = "NameChange"
        old_name = event["predecessors"][0]["name"]
        new_name = event["successors"][0]["name"]
        event.setdefault("changeString", {})["de"] = f"Namensänderung von {old_name} zu {new_name}"
        event.setdefault("changeString", {})["fr"] = f"Changement de nom de {old_name} à {new_name}"
        event.setdefault("changeString", {})["it"] = f"Cambiamento di nome da {old_name} a {new_name}"
        event.setdefault("changeString", {})["en"] = f"Name change from {old_name} to {new_name}"

    # Bezirksumbenennung
    if event["abModes"] == {'22'} and event["adModes"] == {'22'}:
        event["changeType"] = "DistrictRenaming"
        new_name = event["successors"][0]["name"]
        event.setdefault("changeString", {})["de"] = f"Bezirksumbenennung für Gemeinde {new_name}"
        event.setdefault("changeString", {})["fr"] = f"Changement de nom de district pour commune {new_name}"
        event.setdefault("changeString", {})["it"] = f"Cambiamento di nome di distretto per comune {new_name}"
        event.setdefault("changeString", {})["en"] = f"District renaming for municipality {new_name}"

    # Gebietsabtausch
    if event["abModes"] == {'26'} and event["adModes"] == {'26'}:
        event["changeType"] = "TerritorialExchange"
        old = []
        for predecessor in event["predecessors"]:
            old.append(predecessor["name"])
        old = sorted(old)
        event.setdefault("changeString", {})["de"] = f"Gebietsabtausch zwischen {join_names(old, 'de')}"
        event.setdefault("changeString", {})["fr"] = f"Échange territorial entre {join_names(old, 'fr')}"
        event.setdefault("changeString", {})["it"] = f"Scambio territoriale tra {join_names(old, 'it')}"
        event.setdefault("changeString", {})["en"] = f"Territorial exchange between {join_names(old, 'en')}"

    # Ausgemeindung
    if event["abModes"] == {'26'} and event["adModes"] == {'21', '26'}:
        event["changeType"] = "Excommunication"
        new = []
        old = []
        for successor in event["successors"]:
            if successor["mode"] == '21':
                new.append(successor["name"])
            else:
                old.append(successor["name"])
        new = sorted(new)
        old = sorted(old)
        event.setdefault("changeString", {})["de"] = f"Ausgemeindung von {join_names(new, 'de')} aus {join_names(old, 'de')}"
        event.setdefault("changeString", {})["fr"] = f"Excommunication de {join_names(new, 'fr')} de {join_names(old, 'fr')}"
        event.setdefault("changeString", {})["it"] = f"Escomunicazione di {join_names(new, 'it')} da {join_names(old, 'it')}"
        event.setdefault("changeString", {})["en"] = f"Excommunication of {join_names(new, 'en')} from {join_names(old, 'en')}"

    return event

In [8]:
# Enrich event_dict with strings

for event in event_dict:
    event_enriched = create_strings(event_dict[event])
    event_dict[event] = event_enriched

In [9]:
# example

event_dict["4003"]

{'successors': [{'id': '16673', 'name': 'Brugg', 'mode': '26'}],
 'predecessors': [{'id': '10009', 'name': 'Villnachern', 'mode': '29'},
  {'id': '16135', 'name': 'Brugg', 'mode': '26'}],
 'abModes': {'26', '29'},
 'adModes': {'26'},
 'changeType': 'Incorporation',
 'changeString': {'de': 'Eingemeindung von Villnachern in Brugg',
  'fr': 'Incoporation de Villnachern dans Brugg',
  'it': 'Incorporazione di Villnachern in Brugg',
  'en': 'Incorporation of Villnachern into Brugg'}}

## Generate JSON for Events

In [13]:

# copy event_dict to event_json, which will be written to a json file, as json does not support sets, we need to convert the sets to lists
event_json = event_dict

# delete keys

def delete_key(d, key):
    if isinstance(d, dict):
        d.pop(key, None)
        for v in d.values():
            delete_key(v, key)
    elif isinstance(d, list):
        for item in d:
            delete_key(item, key)

# remove abModes (if present) and adModes from event_json as they were only needed to create the strings

delete_key(event_json, "abModes")
delete_key(event_json, "adModes")
delete_key(event_json, "mode")

# write event_json to a json file with indentation and ensure_ascii=False to keep the umlauts
with open("../data/json/events.json", "w", encoding="utf-8") as f:
    import json
    json.dump(event_json, f, ensure_ascii=False, indent=4)

## Create RDF for Change Strings

In [ ]:
from rdflib import Graph, Literal, Namespace, RDF, URIRef

g = Graph(bind_namespaces="none")

asc = Namespace("https://schema.ld.admin.ch/")
g.bind("as", asc)

schema = Namespace("http://schema.org/")
g.bind("schema", schema)

xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)

for event in event_dict:

    # if there is no changeType
    if "changeType" not in event_dict[event]:
        continue

    event_data = event_dict[event]
    event_uri = URIRef(f"https://ld.admin.ch/municipality/changeevent/{event}")
    event_description_uri = URIRef(f"https://ld.admin.ch/municipality/changeevent/{event}/description")
    g.add((event_uri, asc.changeEventDescription, event_description_uri))

    # add id triple (for sorting in visualize.admin.ch)
    if "_" in str(event):
        g.add((event_description_uri, schema.identifier, Literal(1000, datatype=xsd.integer)))
    else:
        g.add((event_description_uri, schema.identifier, Literal(event, datatype=xsd.integer)))

    for lang in event_data.get("changeString", {}):
        g.add((event_description_uri, schema.name, Literal(event_data["changeString"][lang], lang=lang)))   
    
g.serialize("../data/ttl/events.ttl", format="turtle")